# Resume Screening & Role Matching using NLP

**Submitted By:** Kuldeep Panwar | Reg: 24CRD02054

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, re
from collections import Counter
from wordcloud import WordCloud
import missingno as msno

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')
palette = sns.color_palette('Set2')
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

print(f"pandas {pd.__version__} | numpy {np.__version__} | sklearn {__import__('sklearn').__version__}")
print("All libraries imported!")

## Step 2: Load Dataset
Dataset: `parsed_resumes.csv` — 5000 pre-parsed resume records

In [ ]:
# Load CSV — local file path
df_raw = pd.read_csv('parsed_resumes.csv')

print(f"Dataset loaded: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
print()
print("Columns:")
for i, col in enumerate(df_raw.columns, 1):
    print(f"  {i:2}. {col}")
df_raw.head(3)

In [ ]:
# Statistical Summary
print("Statistical Summary:")
df_raw.describe().round(2)

## Step 3: Data Validation & Cleaning
Check missing values, duplicates, logical issues, skill typos

In [ ]:
df = df_raw.copy()

# --- Missing Values ---
missing = df.isnull().sum()
missing_pct = (df.isnull().mean() * 100).round(2)
missing_df = pd.DataFrame({'Count': missing, '%': missing_pct})
missing_df = missing_df[missing_df['Count'] > 0]
print("Missing Values:")
print(missing_df if len(missing_df) > 0 else "  None found!")

# Fix
df['highest_education'].fillna('Not Specified', inplace=True)
df['education_field'].fillna('Not Specified', inplace=True)
print("\nMissing values fixed. Remaining:", df.isnull().sum().sum())

In [ ]:
# Missing value visualization
fig, ax = plt.subplots(figsize=(16, 4))
msno.bar(df_raw, ax=ax, color='steelblue', fontsize=9)
ax.set_title('Data Completeness per Column', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_01_missing_values.png', dpi=150, bbox_inches='tight')
plt.show()
print("Finding: Only education fields have ~5% missing — all other columns 100% complete")

In [ ]:
# --- Duplicates ---
dup = df.duplicated().sum()
print(f"Duplicate rows: {dup}")
if dup > 0:
    df.drop_duplicates(inplace=True)
    print(f"Removed {dup} duplicates")
else:
    print("No duplicates found!")

# --- Logical Inconsistencies ---
junior_high = df[(df['primary_role'].str.contains('Junior', na=False)) & (df['years_experience'] > 8)]
senior_low  = df[(df['primary_role'].str.contains('Senior', na=False)) & (df['years_experience'] < 3)]
print(f"\nLogical check — Junior with >8yr exp: {len(junior_high)}")
print(f"Logical check — Senior with <3yr exp: {len(senior_low)}")

# --- Skill Typos ---
TYPOS = {'Ptyhon': 'Python', 'Pythno': 'Python', 'Pythoon': 'Python', 'Javaa': 'Java'}
all_skills = [s.strip() for t in df['technical_skills_raw'].dropna() for s in t.split(',')]
sc = Counter(all_skills)
print("\nSkill Typos Found:")
for t, c in TYPOS.items():
    cnt = sc.get(t, 0)
    if cnt > 0: print(f"  '{t}' ({cnt}x) → '{c}'")

def fix_typos(text, tmap):
    if pd.isna(text): return text
    for w, r in tmap.items(): text = text.replace(w, r)
    return text

df['technical_skills_clean'] = df['technical_skills_raw'].apply(lambda x: fix_typos(x, TYPOS))
print("\nData validation complete! Clean dataset:", df.shape)

## Step 4: Exploratory Data Analysis (EDA)

In [ ]:
# Domain & Role Distribution
domain_counts = df['primary_domain'].value_counts()
role_counts = df['primary_role'].value_counts()

print("Domain Distribution:")
for d, c in domain_counts.items():
    print(f"  {d}: {c} ({c/len(df)*100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].bar(domain_counts.index, domain_counts.values, color=palette[:len(domain_counts)])
axes[0].set_title('Candidates by Domain', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Domain'); axes[0].set_ylabel('Count')
for i, v in enumerate(domain_counts.values):
    axes[0].text(i, v+10, str(v), ha='center', fontweight='bold')

axes[1].barh(role_counts.head(10).index[::-1], role_counts.head(10).values[::-1], color=palette[2])
axes[1].set_title('Top 10 Job Roles', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.savefig('plot_02_domain_role.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\nKey finding: {domain_counts.index[0]} is dominant domain with {domain_counts.iloc[0]} candidates")

In [ ]:
# Experience Distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(df['years_experience'], bins=20, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(df['years_experience'].mean(), color='red', linestyle='--', lw=2,
                label=f"Mean={df['years_experience'].mean():.1f}yr")
axes[0].axvline(df['years_experience'].median(), color='green', linestyle=':', lw=2,
                label=f"Median={df['years_experience'].median():.0f}yr")
axes[0].set_title('Experience Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Years'); axes[0].set_ylabel('Count'); axes[0].legend()

df.boxplot(column='years_experience', by='primary_domain', ax=axes[1])
axes[1].set_title('Experience by Domain', fontsize=13, fontweight='bold')
plt.suptitle('')
plt.tight_layout()
plt.savefig('plot_03_experience.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Mean experience: {df['years_experience'].mean():.1f} yrs")
print(f"Min: {df['years_experience'].min()} | Max: {df['years_experience'].max()}")

In [ ]:
# Top Skills & Word Cloud
all_tech = [s.strip() for t in df['technical_skills_clean'].dropna() for s in t.split(',')]
tech_freq = Counter(all_tech)
top20 = pd.DataFrame(tech_freq.most_common(20), columns=['Skill', 'Count'])

print("Top 10 Skills:")
for i, r in top20.head(10).iterrows():
    print(f"  {i+1}. {r['Skill']}: {r['Count']}")

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
axes[0].barh(top20['Skill'][::-1], top20['Count'][::-1],
             color=plt.cm.Set3(np.linspace(0, 1, 20)))
axes[0].set_title('Top 20 Technical Skills', fontsize=13, fontweight='bold')

wc = WordCloud(width=600, height=400, background_color='white', colormap='Set2')
wc.generate(' '.join(all_tech))
axes[1].imshow(wc, interpolation='bilinear'); axes[1].axis('off')
axes[1].set_title('Skills Word Cloud', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('plot_04_skills.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Education & Profile Quality
edu_counts  = df['highest_education'].value_counts()
tier_counts = df['institution_tier'].value_counts()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].pie(edu_counts.values, labels=edu_counts.index, autopct='%1.1f%%',
            colors=palette[:len(edu_counts)], startangle=90)
axes[0].set_title('Education Level', fontsize=12, fontweight='bold')

axes[1].bar(tier_counts.index, tier_counts.values, color=[palette[3], palette[4], palette[5]])
axes[1].set_title('Institution Tier', fontsize=12, fontweight='bold')
for i, v in enumerate(tier_counts.values):
    axes[1].text(i, v+10, str(v), ha='center', fontweight='bold')

axes[2].hist(df['profile_completeness_score'], bins=15, color=palette[1], edgecolor='white')
axes[2].axvline(df['profile_completeness_score'].mean(), color='red', linestyle='--', lw=2,
                label=f"Mean={df['profile_completeness_score'].mean():.2f}")
axes[2].set_title('Profile Completeness', fontsize=12, fontweight='bold'); axes[2].legend()

plt.tight_layout()
plt.savefig('plot_05_education.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ATS Flags Analysis
flag_cols   = [c for c in df.columns if c.endswith('_flag')]
flag_labels = [c.replace('_flag','').replace('_',' ').title() for c in flag_cols]
flag_pct    = (df[flag_cols].mean() * 100).values

flag_df = pd.DataFrame({'Flag': flag_labels, 'Pct': flag_pct}).sort_values('Pct', ascending=True)
colors  = ['#2ecc71' if p > 60 else '#e74c3c' if p < 30 else '#f39c12' for p in flag_df['Pct']]

plt.figure(figsize=(14, 7))
plt.barh(flag_df['Flag'], flag_df['Pct'], color=colors)
plt.title('ATS Experience Flags — % of Candidates', fontsize=13, fontweight='bold')
plt.xlabel('% Candidates')
plt.axvline(50, color='navy', linestyle='--', lw=1.5, alpha=0.6)
for i, v in enumerate(flag_df['Pct']):
    plt.text(v+0.5, i, f'{v:.1f}%', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('plot_06_ats_flags.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Total ATS flags: {len(flag_cols)}")
print(f"Most common flag: {flag_df.iloc[-1]['Flag']} ({flag_df.iloc[-1]['Pct']:.1f}%)")

## Step 5: Feature Engineering
Create skill match scores, experience buckets, domain alignment, education score

In [ ]:
# Job Description for target role
TARGET_ROLE = 'Data Scientist'
JD = {
    'mandatory': ['Python', 'Machine Learning', 'SQL', 'Statistics'],
    'optional' : ['TensorFlow', 'NLP', 'Power BI', 'Deep Learning'],
    'min_exp'  : 3,
    'domain'   : 'Data Science',
    'jd_text'  : 'python machine learning sql statistics tensorflow nlp pandas scikit learn data analysis feature engineering model training prediction'
}

SKILL_MAP = {
    'ml': 'Machine Learning', 'ptyhon': 'Python', 'pythno': 'Python',
    'python': 'Python', 'sql': 'SQL', 'mysql': 'SQL',
    'aws': 'AWS', 'k8s': 'Kubernetes', 'tensorflow': 'TensorFlow',
    'sklearn': 'Scikit-Learn', 'nlp': 'NLP', 'js': 'JavaScript'
}

def normalize_skills(text):
    if pd.isna(text): return []
    skills = [s.strip() for s in text.split(',')]
    return [SKILL_MAP.get(s.lower().strip(), s.strip()) for s in skills]

def skill_match(candidate_skills, required):
    if not required: return 0.0
    c = [s.lower() for s in candidate_skills]
    return round(sum(1 for s in required if s.lower() in c) / len(required) * 100, 1)

# Build skill features
df['skills_norm'] = df['technical_skills_clean'].apply(normalize_skills)
df['tools_norm']  = df['tools_platforms_raw'].apply(normalize_skills)
df['all_skills']  = df['skills_norm'] + df['tools_norm']

df['mandatory_match_pct'] = df['all_skills'].apply(lambda s: skill_match(s, JD['mandatory']))
df['optional_match_pct']  = df['all_skills'].apply(lambda s: skill_match(s, JD['optional']))
df['skill_score']         = (df['mandatory_match_pct']*0.70 + df['optional_match_pct']*0.30).round(1)

print(f"Target Role: {TARGET_ROLE}")
print(f"Mandatory skills: {JD['mandatory']}")
print()
print("Skill match results (sample):")
print(df[['candidate_name','mandatory_match_pct','optional_match_pct','skill_score']].head(5).to_string(index=False))

In [ ]:
# Experience, Domain, Education features
def exp_bucket(y):
    if y <= 2: return 'Junior'
    elif y <= 5: return 'Mid'
    elif y <= 10: return 'Senior'
    else: return 'Lead'

df['experience_bucket']  = df['years_experience'].apply(exp_bucket)
df['experience_score']   = df['years_experience'].apply(lambda y: min(100, round(y/JD['min_exp']*100, 1)))
df['seniority_align']    = ((df['experience_bucket']=='Senior')|(df['experience_bucket']=='Mid')).astype(int)
df['domain_match']       = (df['primary_domain']==JD['domain']).astype(int)
df['domain_match_pct']   = df['domain_match'] * 100

edu_map  = {'Masters':100,'MBA':90,'LLM':80,'Bachelors':70,'Not Specified':50}
tier_map = {'Tier-1':10,'Tier-2':5,'Tier-3':0}
df['edu_score']     = df['highest_education'].map(edu_map).fillna(50)
df['tier_bonus']    = df['institution_tier'].map(tier_map).fillna(0)
df['edu_total']     = (df['edu_score'] + df['tier_bonus']).clip(upper=100)

lead_flags = ['management_experience_flag','people_management_flag',
              'delivery_lead_experience_flag','mentoring_experience_flag',
              'stakeholder_management_experience_flag']
df['leadership_score'] = (df[lead_flags].sum(axis=1)/len(lead_flags)*100).round(1)

all_flags = [c for c in df.columns if c.endswith('_flag')]
df['ats_score'] = (df[all_flags].sum(axis=1)/len(all_flags)*100).round(1)

# Rule-based fit score
df['rule_score'] = (
    df['skill_score']    * 0.40 +
    df['experience_score']* 0.25 +
    df['domain_match_pct']* 0.15 +
    df['edu_total']      * 0.10 +
    df['ats_score']      * 0.10
).clip(0,100).round(2)

def fit_label(s):
    if s >= 70: return 'High Fit'
    elif s >= 45: return 'Medium Fit'
    else: return 'Low Fit'

df['fit_label'] = df['rule_score'].apply(fit_label)

print("Feature engineering complete!")
print(f"New features added: {df.shape[1] - df_raw.shape[1]}")
print()
print("Fit Label Distribution (Rule-Based):")
print(df['fit_label'].value_counts().to_string())
print()
print("Experience Buckets:")
print(df['experience_bucket'].value_counts().to_string())

## Step 6: NLP — TF-IDF Similarity Scoring
Compute cosine similarity between resume text and job description

In [ ]:
def preprocess(text):
    if pd.isna(text) or not text.strip(): return ''
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    try: tokens = word_tokenize(text)
    except: tokens = text.split()
    return ' '.join([lemmatizer.lemmatize(t) for t in tokens
                     if t not in stop_words and len(t) > 2])

# Combine resume text fields
df['resume_text'] = (
    df['technical_skills_clean'].fillna('') + ' ' +
    df['experience_summary'].fillna('')     + ' ' +
    df['project_summary'].fillna('')        + ' ' +
    df['key_achievements'].fillna('')
)
df['resume_clean'] = df['resume_text'].apply(preprocess)

jd_clean = preprocess(JD['jd_text'])
print(f"JD text (processed): {jd_clean}")
print(f"\nAvg resume words: {df['resume_clean'].str.split().str.len().mean():.0f}")

# TF-IDF
vectorizer  = TfidfVectorizer(max_features=1000, ngram_range=(1,2), min_df=2)
all_texts   = df['resume_clean'].tolist() + [jd_clean]
tfidf_mat   = vectorizer.fit_transform(all_texts)

sim_scores          = cosine_similarity(tfidf_mat[:-1], tfidf_mat[-1]).flatten()
df['tfidf_sim']     = (sim_scores * 100).round(2)
tfidf_max           = df['tfidf_sim'].max()
df['tfidf_norm']    = (df['tfidf_sim'] / tfidf_max * 100).round(2) if tfidf_max > 0 else 0

# Hybrid Score
df['hybrid_score'] = (df['rule_score']*0.60 + df['tfidf_norm']*0.40).clip(0,100).round(2)
df['hybrid_label'] = df['hybrid_score'].apply(
    lambda s: 'High Fit' if s>=65 else 'Medium Fit' if s>=40 else 'Low Fit')

print(f"\nVocabulary size: {len(vectorizer.vocabulary_):,} terms")
print(f"Similarity range: {df['tfidf_sim'].min():.2f}% - {df['tfidf_sim'].max():.2f}%")
print()
print("Hybrid Fit Distribution:")
print(df['hybrid_label'].value_counts().to_string())

In [ ]:
# Score distribution chart
plt.figure(figsize=(12, 5))
plt.hist(df['hybrid_score'], bins=30, color='steelblue', edgecolor='white', alpha=0.85)
plt.axvline(65, color='green',  linestyle='--', lw=2.5, label='High Fit (≥65)')
plt.axvline(40, color='orange', linestyle='--', lw=2.5, label='Medium Fit (≥40)')
plt.title('Hybrid Fit Score Distribution', fontsize=13, fontweight='bold')
plt.xlabel('Score'); plt.ylabel('Count'); plt.legend()
plt.tight_layout()
plt.savefig('plot_07_score_dist.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Mean score: {df['hybrid_score'].mean():.2f}")

## Step 7: ML Modeling & Evaluation
Train 3 classifiers, compare accuracy, confusion matrix, feature importance

In [ ]:
feat_cols = [
    'years_experience','keyword_density_score','profile_completeness_score','resume_length_words',
    'mandatory_match_pct','optional_match_pct','skill_score',
    'experience_score','seniority_align','domain_match',
    'edu_score','tier_bonus','leadership_score','ats_score','tfidf_norm',
    'management_experience_flag','project_management_experience_flag',
    'agile_scrum_experience_flag','client_facing_experience_flag',
    'cloud_experience_flag','ml_experience_flag','stakeholder_management_experience_flag'
]

le = LabelEncoder()
y  = le.fit_transform(df['hybrid_label'])
X  = df[feat_cols].fillna(0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler     = StandardScaler()
X_tr_sc    = scaler.fit_transform(X_train)
X_te_sc    = scaler.transform(X_test)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest'       : RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting'   : GradientBoostingClassifier(n_estimators=100, random_state=42)
}

print(f"Train: {len(X_train)} | Test: {len(X_test)} | Classes: {list(le.classes_)}\n")

results, trained = {}, {}
for name, model in models.items():
    xtr = X_tr_sc if 'Logistic' in name else X_train
    xte = X_te_sc if 'Logistic' in name else X_test
    model.fit(xtr, y_train)
    preds = model.predict(xte)
    acc   = accuracy_score(y_test, preds)
    results[name] = acc
    trained[name] = (model, preds)
    print(f"  {name:<28}: {acc*100:.2f}%")

best_name = max(results, key=results.get)
print(f"\nBest Model: {best_name} ({results[best_name]*100:.2f}%)")

In [ ]:
best_model, best_preds = trained[best_name]

print(f"=== {best_name} — Classification Report ===")
print(classification_report(y_test, best_preds, target_names=le.classes_))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

cm = confusion_matrix(y_test, best_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=axes[0],
            annot_kws={'size':13,'weight':'bold'})
axes[0].set_title(f'Confusion Matrix — {best_name}', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Actual'); axes[0].set_xlabel('Predicted')

accs = [v*100 for v in results.values()]
bars = axes[1].bar(list(results.keys()), accs, color=['#e74c3c','#3498db','#2ecc71'])
axes[1].set_title('Model Accuracy Comparison', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Accuracy (%)'); axes[1].set_ylim(0, 110)
for bar, a in zip(bars, accs):
    axes[1].text(bar.get_x()+bar.get_width()/2, a+1, f'{a:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('plot_08_model_eval.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature Importance
if hasattr(best_model, 'feature_importances_'):
    fi = pd.DataFrame({'Feature': feat_cols, 'Importance': best_model.feature_importances_})
    fi = fi.sort_values('Importance', ascending=True).tail(15)

    plt.figure(figsize=(12, 7))
    plt.barh(fi['Feature'], fi['Importance'], color='steelblue')
    plt.title(f'Top 15 Feature Importances — {best_name}', fontsize=13, fontweight='bold')
    plt.xlabel('Importance Score')
    plt.tight_layout()
    plt.savefig('plot_09_feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()

    print("Top 5 features driving fit score:")
    for i, (_, r) in enumerate(fi.tail(5).iloc[::-1].iterrows(), 1):
        print(f"  {i}. {r['Feature']}: {r['Importance']:.4f}")

## Step 8: Final Rankings & Business Recommendations

In [ ]:
def explain(row):
    parts = []
    m = row.get('mandatory_match_pct', 0)
    parts.append(f"Skills:{m:.0f}%({'STRONG' if m>=75 else 'PARTIAL' if m>=50 else 'WEAK'})")
    y = row.get('years_experience', 0)
    parts.append(f"Exp:{y}yr({'OK' if y>=JD['min_exp'] else 'BELOW MIN'})")
    parts.append('Domain:MATCH' if row.get('domain_match',0)==1 else 'Domain:MISMATCH')
    t = row.get('tfidf_norm', 0)
    parts.append(f"NLP:{t:.0f}%({'HIGH' if t>=50 else 'LOW'})")
    return ' | '.join(parts)

ranked = df.sort_values('hybrid_score', ascending=False).reset_index(drop=True)
ranked['rank']    = ranked.index + 1
ranked['explain'] = ranked.apply(explain, axis=1)

print(f"TOP 20 CANDIDATES FOR: {TARGET_ROLE}")
print('='*90)
print(ranked[['rank','candidate_name','years_experience',
              'mandatory_match_pct','hybrid_score','hybrid_label']].head(20).to_string(index=False))

In [ ]:
# Final visualization dashboard
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Score distribution
axes[0].hist(df['hybrid_score'], bins=30, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(65, color='green',  linestyle='--', lw=2, label='High (≥65)')
axes[0].axvline(40, color='orange', linestyle='--', lw=2, label='Medium (≥40)')
axes[0].set_title('Fit Score Distribution', fontsize=12, fontweight='bold')
axes[0].legend()

# Fit label pie
lc = df['hybrid_label'].value_counts()
cmap = {'High Fit':'#2ecc71','Medium Fit':'#f39c12','Low Fit':'#e74c3c'}
axes[1].pie(lc.values, labels=lc.index, autopct='%1.1f%%',
            colors=[cmap.get(l,'gray') for l in lc.index], startangle=90)
axes[1].set_title('Candidate Fit Breakdown', fontsize=12, fontweight='bold')

# Top 10
top10 = ranked.head(10)
axes[2].barh(top10['candidate_name'][::-1], top10['hybrid_score'][::-1],
             color=[cmap.get(l,'gray') for l in top10['hybrid_label'][::-1]])
axes[2].set_title('Top 10 Candidates', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Hybrid Fit Score')
axes[2].axvline(65, color='green', linestyle='--', lw=1.5, alpha=0.7)

plt.suptitle(f'Resume Screening Results — {TARGET_ROLE}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_10_final_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Business Recommendations
high   = df[df['hybrid_label']=='High Fit']
medium = df[df['hybrid_label']=='Medium Fit']
low    = df[df['hybrid_label']=='Low Fit']
reduction = (1 - (len(high)+len(medium))/len(df)) * 100

print('='*65)
print('  BUSINESS RECOMMENDATIONS — TalentBridge Solutions')
print(f'  Role: {TARGET_ROLE}')
print('='*65)
print(f'Total Screened : {len(df):,}')
print(f'High Fit       : {len(high):,} ({len(high)/len(df)*100:.1f}%) → SHORTLIST')
print(f'Medium Fit     : {len(medium):,} ({len(medium)/len(df)*100:.1f}%) → HUMAN REVIEW')
print(f'Low Fit        : {len(low):,} ({len(low)/len(df)*100:.1f}%) → REJECT')
print()
print(f'Recruiter Workload Reduction: {reduction:.1f}%')
print()
print('Key Shortlisting Thresholds:')
print('  ≥ 65% = High Fit   → Auto shortlist')
print('  40-64% = Medium Fit → Manual review')
print('  < 40%  = Low Fit   → Reject')
print()
print(f'Best ML Model: {best_name} | Accuracy: {results[best_name]*100:.2f}%')

In [ ]:
# Save output files
final_out = ranked[[
    'rank','candidate_id','candidate_name','primary_role','primary_domain',
    'years_experience','highest_education','mandatory_match_pct',
    'optional_match_pct','tfidf_norm','hybrid_score','hybrid_label','explain'
]].rename(columns={
    'mandatory_match_pct':'mandatory_skill_%',
    'optional_match_pct' :'optional_skill_%',
    'tfidf_norm'         :'nlp_similarity',
    'hybrid_score'       :'final_fit_score',
    'hybrid_label'       :'recommendation',
    'explain'            :'explanation'
})

df.to_csv('resumes_cleaned_features.csv', index=False)
final_out.to_csv('final_candidate_rankings.csv', index=False)

print("Files saved:")
print(f"  resumes_cleaned_features.csv  — {len(df):,} rows, {df.shape[1]} columns")
print(f"  final_candidate_rankings.csv  — {len(final_out):,} ranked candidates")
print()
print("PROJECT COMPLETE! All deliverables ready for submission.")